# Persona C — CNP, ANP, TNP y GP Matérn en glucosa (Colab T4)

Notebook para entrenar y evaluar los modelos de C con los mismos datos, tareas y métricas que FlowNP. Antes de empezar, leer `PROYECTO/CONTEXTO_PERSONA_C.md`.

**Orden:** 1 (sesión) → 2 (estado) → 3 (entrenar) → 4 (evaluar) → 5 (GP) → 6 (E4 de TNP) → 7 (resumen) → 8 (figuras) → 9 (cerrar).

**En cada sesión nueva:** Entorno de ejecución → Cambiar tipo de entorno → **T4 GPU**, y ejecutar las celdas de la sección 1. Todo es reanudable: el entrenamiento continúa desde el último checkpoint y la evaluación salta lo que ya está en `results/`.

**Cuota de GPU:** la gratuita se agota tras unas 3–4 h seguidas. Reparte el trabajo con las listas `MODELS` y `SEEDS` (por ejemplo, un modelo por día, o semillas en otra cuenta de Google con la carpeta compartida).

## 1. Preparar la sesión

In [7]:
from google.colab import drive
drive.mount('/content/drive')
PROJ = '/content/drive/MyDrive/PROYECTO'   # <- acceso directo a la carpeta compartida
%cd {PROJ}/flowNP
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/.shortcut-targets-by-id/1-A31mncyUSLGOcnWnTMpBI0GYCHOhs7P/PROYECTO/flowNP
name, memory.used [MiB], memory.total [MiB]
Tesla T4, 0 MiB, 15360 MiB


Comprobación: todos los ficheros de C deben salir como `OK`. Si alguno sale como `FALTA`, no está en su carpeta de Drive (ver `CONTEXTO_PERSONA_C.md`).

In [8]:
import os
for f in ['baselines.py', 'gp_matern.py', 'glucose.py', 'models/tnpd.py',
          'configs/glucose/cnp.yaml', 'configs/glucose/anp.yaml', 'configs/glucose/canp.yaml',
          'configs/glucose/tnpa.yaml', 'configs/glucose/tnpd.yaml']:
    print(('OK    ' if os.path.exists(f) else 'FALTA ') + f)

OK    baselines.py
OK    gp_matern.py
OK    glucose.py
OK    models/tnpd.py
OK    configs/glucose/cnp.yaml
OK    configs/glucose/anp.yaml
OK    configs/glucose/canp.yaml
OK    configs/glucose/tnpa.yaml
OK    configs/glucose/tnpd.yaml


In [5]:
!pip -q install attrdictionary

## 2. Estado: qué está entrenado y evaluado

In [9]:
import os, json, glob
TASKS = ['e1_test', 'e2_nc5', 'e2_nc10', 'e2_nc20', 'e2_nc50',
         'e3_gap30', 'e3_gap60', 'e3_gap120', 'e3_gap180', 'e3_fc30', 'e3_fc60']
# modelo del script -> nombre en glucose_np/results/
NAMES = {'cnp': 'CNP', 'anp': 'ANP', 'tnpa': 'TNP', 'tnpd': 'TNP-D', 'canp': 'CANP', 'gp': 'GP'}
CK = 'output/results/glucose/default'
RES = '../glucose_np/results'
for m, name in NAMES.items():
    for seed in [0, 1, 2]:
        model_file = 'hyper.json' if m == 'gp' else 'best.tar'
        ok = os.path.exists(f'{CK}/{m}/seed{seed}/{model_file}')
        done = [t for t in TASKS if os.path.exists(f'{RES}/{name}/seed{seed}/{t}.json')]
        if not ok and not done:
            continue
        missing = [t for t in TASKS if t not in done]
        print(f'{name:6s} seed{seed}: modelo {"OK" if ok else "FALTA"} | evaluados {len(done)}/11'
              + (f' | faltan: {", ".join(missing)}' if missing and len(missing) < 11 else ''))
for name in ['FlowNP', 'TNP']:
    print(f'E4 {name}:', 'OK' if os.path.exists(f'{RES}/{name}/e4_timing.json') else 'FALTA')

CNP    seed0: modelo OK | evaluados 11/11
CNP    seed1: modelo OK | evaluados 11/11
CNP    seed2: modelo OK | evaluados 11/11
ANP    seed0: modelo OK | evaluados 11/11
ANP    seed1: modelo OK | evaluados 11/11
ANP    seed2: modelo OK | evaluados 11/11
TNP    seed0: modelo OK | evaluados 11/11
TNP    seed1: modelo OK | evaluados 11/11
TNP    seed2: modelo OK | evaluados 4/11 | faltan: e2_nc50, e3_gap30, e3_gap60, e3_gap120, e3_gap180, e3_fc30, e3_fc60
E4 FlowNP: OK
E4 TNP: OK


## 3. Entrenamiento

100k pasos por modelo y semilla, como FlowNP (Adam 5e-4 con cosine, batch 16). El mejor checkpoint (`best.tar`) se elige por la log-verosimilitud media en `e1_val`. Se guarda un checkpoint cada 1.000 pasos: si la sesión se corta, volver a ejecutar esta celda y **continúa donde iba**.

Modelos (`MODELS`):
- `cnp` → **CNP**
- `anp` → **ANP** (con camino latente; muestras de z)
- `tnpa` → **TNP** (autorregresivo; es el que se compara en tiempo con FlowNP, H3)
- opcionales: `tnpd` → TNP-D (TNP gaussiano de una pasada), `canp` → CANP (ANP sin latente)

Tiempos orientativos en T4 (no medidos): CNP ~15–20 min, ANP ~30 min, TNP ~35 min por semilla. FlowNP tardó ~33 min.

In [10]:
MODELS = ['cnp', 'tnpa','anp']
SEEDS = [0, 1, 2]
for m in MODELS:
    for seed in SEEDS:
        resume = '--resume' if os.path.exists(f'{CK}/{m}/seed{seed}/ckpt.tar') else ''
        print(f'===== {m} seed{seed} {resume}')
        !python baselines.py --model={m} --mode=train --seed={seed} {resume}

===== cnp seed0 --resume
Using device: cuda
===== cnp seed1 --resume
Using device: cuda
===== cnp seed2 --resume
Using device: cuda
===== tnpa seed0 --resume
Using device: cuda
===== tnpa seed1 --resume
Using device: cuda
===== tnpa seed2 --resume
Using device: cuda
===== anp seed0 --resume
Using device: cuda
===== anp seed1 --resume
Using device: cuda
===== anp seed2 --resume
Using device: cuda


In [12]:
# Progreso: últimas líneas de validación de cada entrenamiento
for m in ['cnp', 'anp', 'tnpa', 'tnpd', 'canp']:
    for seed in [0, 1, 2]:
        logs = sorted(glob.glob(f'{CK}/{m}/seed{seed}/train_*.log'))
        if logs:
            val = [l.strip() for l in open(logs[-1]) if 'val_ll' in l]
            print(f'{m} seed{seed}:', val[-1] if val else '(sin validación todavía)')

cnp seed0: glucose:cnp:seed0 step 100000 [val_ll] 0.55510
cnp seed1: glucose:cnp:seed1 step 100000 [val_ll] 0.49848
cnp seed2: glucose:cnp:seed2 step 100000 [val_ll] 0.56298
anp seed0: glucose:anp:seed0 step 100000 [val_ll] 1.21985 (best)
anp seed1: glucose:anp:seed1 step 100000 [val_ll] 1.22634 (best)
anp seed2: glucose:anp:seed2 step 100000 [val_ll] 1.23321 (best)
tnpa seed0: glucose:tnpa:seed0 step 100000 [val_ll] 1.99660
tnpa seed1: glucose:tnpa:seed1 step 100000 [val_ll] 2.01432
tnpa seed2: glucose:tnpa:seed2 step 100000 [val_ll] 1.99736


## 4. Evaluación (11 experimentos por modelo y semilla)

Escribe `glucose_np/results/<CNP|ANP|TNP>/seed<k>/<tarea>.json`. Salta lo que ya existe.
- CNP / TNP-D / CANP: predictiva gaussiana (`add_gaussian`). Muy rápido.
- ANP: 100 muestras de z con ruido de observación; LL por punto con logmeanexp sobre z.
- TNP: 100 muestras autorregresivas (una pasada por punto objetivo; es lo lento) y LL autorregresiva conjunta con teacher forcing. Usa batch 32 para no quedarse sin memoria.

In [13]:
MODELS = ['cnp', 'anp', 'tnpa']
SEEDS = [0, 1, 2]
EVAL_BS = {'tnpa': 32}          # el resto usa 64
for m in MODELS:
    for seed in SEEDS:
        print(f'===== {m} seed{seed}')
        !python baselines.py --model={m} --mode=eval --seed={seed} --eval_batch_size={EVAL_BS.get(m, 64)}

===== cnp seed0
Using device: cuda
Loaded best.tar (step 96000, val_ll 0.55659)
Command line: baselines.py --model=cnp --mode=eval --seed=0 --eval_batch_size=64
e1_test    already in /content/drive/.shortcut-targets-by-id/1-A31mncyUSLGOcnWnTMpBI0GYCHOhs7P/PROYECTO/glucose_np/results/CNP/seed0/e1_test.json, skipping (use --overwrite)
e2_nc5     already in /content/drive/.shortcut-targets-by-id/1-A31mncyUSLGOcnWnTMpBI0GYCHOhs7P/PROYECTO/glucose_np/results/CNP/seed0/e2_nc5.json, skipping (use --overwrite)
e2_nc10    already in /content/drive/.shortcut-targets-by-id/1-A31mncyUSLGOcnWnTMpBI0GYCHOhs7P/PROYECTO/glucose_np/results/CNP/seed0/e2_nc10.json, skipping (use --overwrite)
e2_nc20    already in /content/drive/.shortcut-targets-by-id/1-A31mncyUSLGOcnWnTMpBI0GYCHOhs7P/PROYECTO/glucose_np/results/CNP/seed0/e2_nc20.json, skipping (use --overwrite)
e2_nc50    already in /content/drive/.shortcut-targets-by-id/1-A31mncyUSLGOcnWnTMpBI0GYCHOhs7P/PROYECTO/glucose_np/results/CNP/seed0/e2_nc50.jso

## 5. GP Matérn 5/2

Hiperparámetros globales (lengthscale, varianza, ruido y media) ajustados por máxima verosimilitud marginal en ventanas de train; cada semilla usa una submuestra distinta. Después, GP exacto condicionado al contexto de cada tarea. Tarda unos minutos y **no necesita GPU**.

In [14]:
for seed in [0, 1, 2]:
    if not os.path.exists(f'{CK}/gp/seed{seed}/hyper.json'):
        !python gp_matern.py --mode=fit --seed={seed}
    !python gp_matern.py --mode=eval --seed={seed}

Using device: cuda
step 100 mll/point 0.6702 | l 0.3055 (55 min) s2 0.515 noise 0.00187 mean -0.014
step 200 mll/point 0.8006 | l 0.2999 (54 min) s2 0.670 noise 0.00143 mean 0.024
step 300 mll/point 0.7533 | l 0.3002 (54 min) s2 0.671 noise 0.00143 mean -0.005
step 400 mll/point 0.7281 | l 0.3040 (55 min) s2 0.693 noise 0.00146 mean -0.020
step 500 mll/point 0.8824 | l 0.3028 (55 min) s2 0.685 noise 0.00136 mean 0.021
step 600 mll/point 0.8077 | l 0.3072 (55 min) s2 0.667 noise 0.00140 mean 0.013
step 700 mll/point 0.7804 | l 0.3029 (55 min) s2 0.681 noise 0.00146 mean -0.009
step 800 mll/point 0.8149 | l 0.3029 (55 min) s2 0.688 noise 0.00153 mean -0.001
step 900 mll/point 0.6170 | l 0.2923 (53 min) s2 0.701 noise 0.00153 mean 0.048
step 1000 mll/point 0.7780 | l 0.2964 (53 min) s2 0.698 noise 0.00147 mean 0.007
Saved output/results/glucose/default/gp/seed0/hyper.json
Using device: cuda
Loaded hyper.json: l 0.2964 s2 0.698 noise 0.00147 mean 0.007
e1_test: 100% 518/518 [00:02<00:00, 2

## 6. E4 de TNP (hipótesis H3)

Mismas condiciones que FlowNP: **T4**, `e1_test`, **100 muestras**, **batch 16**, tiempo de muestreo puro. Guarda `glucose_np/results/TNP/e4_timing.json` con `steps` = número medio de puntos objetivo (≈ 42 pasadas secuenciales), que es como aparece en la figura de E4 junto a la curva de FlowNP. Comprueba con `nvidia-smi` (celda 1) que la GPU es una T4. Tarda unos 15–30 min.

In [7]:
!python baselines.py --model=tnpa --mode=e4 --seed=0 --eval_batch_size=16 --num_samples=100
print(open('../glucose_np/results/TNP/e4_timing.json').read())

Using device: cuda
Loaded best.tar (step 90000, val_ll 2.00485)
Command line: baselines.py --model=tnpa --mode=e4 --seed=0 --eval_batch_size=16 --num_samples=100
e4: 100% 556/556 [15:09<00:00,  1.64s/it]
{"steps": 42.1, "method": "tnpa", "nfe": 42.1, "time_per_task_s": 0.471559855596257, "crps_mgdl": 3.862603523999603, "rmse_mgdl": 11.059658273409193, "cov90": 0.9317106319886328, "loglik_mgdl": null, "num_samples": 100, "batch_size": 16, "n_tasks": 1922, "mean_nt": 42.10978147762747, "device": "Tesla T4"}
{
  "model": "TNP",
  "rows": [
    {
      "steps": 42.1,
      "method": "tnpa",
      "nfe": 42.1,
      "time_per_task_s": 0.471559855596257,
      "crps_mgdl": 3.862603523999603,
      "rmse_mgdl": 11.059658273409193,
      "cov90": 0.9317106319886328,
      "loglik_mgdl": null,
      "num_samples": 100,
      "batch_size": 16,
      "n_tasks": 1922,
      "mean_nt": 42.10978147762747,
      "device": "Tesla T4"
    }
  ]
}


In [8]:
# Comparación directa con FlowNP (misma GPU, mismas condiciones)
fl = json.load(open('output/results/glucose/default/fnp/seed0/e4_all.json'))
tn = json.load(open('../glucose_np/results/TNP/e4_timing.json'))['rows'][0]
print(f"TNP     {tn['steps']:6.1f} pasadas  {tn['time_per_task_s']*1000:8.1f} ms/tarea  CRPS {tn['crps_mgdl']:.2f}")
for method, rows in (fl.items() if isinstance(fl, dict) else [('', fl)]):
    for r in rows:
        print(f"FlowNP  {method:8s} {r['steps']:3d} pasos ({r['nfe']:3d} NFE)  "
              f"{r['time_per_task_s']*1000:8.1f} ms/tarea  CRPS {r['crps_mgdl']:.2f}")

TNP       42.1 pasadas     471.6 ms/tarea  CRPS 3.86
FlowNP  euler      1 pasos (  1 NFE)       5.2 ms/tarea  CRPS 4.43
FlowNP  euler      2 pasos (  2 NFE)      10.8 ms/tarea  CRPS 4.06
FlowNP  euler      5 pasos (  5 NFE)      29.2 ms/tarea  CRPS 3.78
FlowNP  euler     10 pasos ( 10 NFE)      60.3 ms/tarea  CRPS 3.66
FlowNP  euler     20 pasos ( 20 NFE)     121.2 ms/tarea  CRPS 3.60
FlowNP  euler     50 pasos ( 50 NFE)     303.5 ms/tarea  CRPS 3.59
FlowNP  euler    100 pasos (100 NFE)     607.8 ms/tarea  CRPS 3.59
FlowNP  euler    200 pasos (200 NFE)    1215.0 ms/tarea  CRPS 3.60
FlowNP  midpoint   1 pasos (  2 NFE)      11.7 ms/tarea  CRPS 3.84
FlowNP  midpoint   2 pasos (  4 NFE)      23.8 ms/tarea  CRPS 3.70
FlowNP  midpoint   5 pasos ( 10 NFE)      60.4 ms/tarea  CRPS 3.61
FlowNP  midpoint  10 pasos ( 20 NFE)     121.1 ms/tarea  CRPS 3.59
FlowNP  midpoint  20 pasos ( 40 NFE)     241.9 ms/tarea  CRPS 3.59
FlowNP  midpoint  50 pasos (100 NFE)     607.3 ms/tarea  CRPS 3.60


## 7. Resumen de resultados (media ± desviación entre semillas)

In [15]:
import numpy as np
KEYS = [('loglik_mgdl', 'LL'), ('rmse_mgdl', 'RMSE'), ('mae_mgdl', 'MAE'), ('crps_mgdl', 'CRPS'), ('cov90', 'cov90')]
ALL = ['FlowNP', 'TNP', 'TNP-D', 'ANP', 'CANP', 'CNP', 'GP', 'Interp. lineal', 'Media contexto']
print(f"{'experimento':10s} {'modelo':15s} {'n':>2s} " + ' '.join(f'{n:>14s}' for _, n in KEYS))
for t in TASKS:
    for m in ALL:
        fs = sorted(glob.glob(f'../glucose_np/results/{m}/seed*/{t}.json'))
        if not fs:
            continue
        o = [json.load(open(f))['overall'] for f in fs]
        cols = [f'{np.mean([x[k] for x in o]):7.3f}±{np.std([x[k] for x in o]):5.3f}' for k, _ in KEYS]
        print(f'{t:10s} {m:15s} {len(fs):2d} ' + ' '.join(f'{c:>14s}' for c in cols))
    print()

experimento modelo           n             LL           RMSE            MAE           CRPS          cov90
e1_test    FlowNP           3   -2.190±0.006   10.588±0.071    4.852±0.004    3.585±0.001    0.890±0.007
e1_test    TNP              3   -2.269±0.003   10.984±0.041    5.113±0.022    3.831±0.023    0.932±0.001
e1_test    ANP              3   -3.065±0.004   10.397±0.049    4.992±0.008    3.815±0.010    0.924±0.002
e1_test    CNP              3   -3.745±0.035   14.611±0.324    8.946±0.271    6.560±0.192    0.899±0.003
e1_test    GP               3   -3.306±0.002   12.882±0.105    5.987±0.035    4.704±0.033    0.939±0.001
e1_test    Interp. lineal   3   -3.818±0.000   11.076±0.005    5.543±0.001    5.776±0.001    0.991±0.000
e1_test    Media contexto   3   -4.761±0.000   40.844±0.000   28.039±0.000   19.223±0.000    0.894±0.000

e2_nc5     FlowNP           3   -3.152±0.005   25.535±0.684   13.769±0.197   10.527±0.158    0.940±0.005
e2_nc5     TNP              3   -3.422±0.027   24.079

## 8. Tablas y figuras (código de A)
Salen en `glucose_np/tables/` y `glucose_np/figures/`, con todos los modelos que haya en `results/`.

In [16]:
%cd {PROJ}/glucose_np
!python plots.py
%cd {PROJ}/flowNP

/content/drive/.shortcut-targets-by-id/1-A31mncyUSLGOcnWnTMpBI0GYCHOhs7P/PROYECTO/glucose_np
Resultados encontrados: 77 (modelo, tarea) | modelos: ['FlowNP', 'TNP', 'ANP', 'CNP', 'GP', 'Interp. lineal', 'Media contexto']
Tablas en /content/drive/.shortcut-targets-by-id/1-A31mncyUSLGOcnWnTMpBI0GYCHOhs7P/PROYECTO/glucose_np/tables
Figuras en /content/drive/.shortcut-targets-by-id/1-A31mncyUSLGOcnWnTMpBI0GYCHOhs7P/PROYECTO/glucose_np/figures
/content/drive/.shortcut-targets-by-id/1-A31mncyUSLGOcnWnTMpBI0GYCHOhs7P/PROYECTO/flowNP


## 9. Antes de cerrar
Para la celda que esté corriendo (■) y ejecuta esto para que Drive termine de guardar todo.

In [17]:
from google.colab import drive
drive.flush_and_unmount()